# Stratified DFT Hessians

Open this notebook from GitHub in a fresh GPU runtime. Before running any cell, use the left **Files** panel to upload `oa_audit_02_horm_screen_generation_8x8_r2_j2.tar.gz` into `/content`. It starts with two GPU4PySCF cases to calibrate wall time.

In [ ]:
from pathlib import Path

INPUT_BUNDLE = Path('/content/oa_audit_02_horm_screen_generation_8x8_r2_j2.tar.gz')
assert INPUT_BUNDLE.is_file(), 'Upload the stage-02 archive to /content with the left Files panel first.'
print(f'Input bundle: {INPUT_BUNDLE.stat().st_size / 1024**2:.2f} MiB')

In [ ]:
from pathlib import Path
import os
import tarfile

input_bundle = INPUT_BUNDLE
OUTPUT_ROOT = Path('/content/oa_audit_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
output_root_resolved = OUTPUT_ROOT.resolve()
with tarfile.open(input_bundle, 'r:gz') as archive:
    for member in archive.getmembers():
        target = (OUTPUT_ROOT / member.name).resolve()
        assert target == output_root_resolved or output_root_resolved in target.parents
        assert member.isfile() or member.isdir(), f'Unsupported archive entry: {member.name}'
    archive.extractall(OUTPUT_ROOT)

REPOSITORY_URL = 'https://github.com/jiaxi98/OAReactDiff.git'
REPOSITORY_REF = 'agent/oa-failure-audit'
REPO = Path('/content/OAReactDiff')
MAMBA = '/usr/local/bin/micromamba'
ENV_PREFIX = Path('/content/micromamba/envs/oa-dft')
os.environ['LD_LIBRARY_PATH'] = f"{ENV_PREFIX}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')
SCREEN_ROOT = OUTPUT_ROOT / 'horm_screen_generation_8x8_r2_j2'
SUBSET = SCREEN_ROOT / 'dft_subset.csv'
assert SUBSET.is_file(), f'The uploaded stage-02 bundle is missing {SUBSET}'
if not (REPO / '.git').is_dir():
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {REPOSITORY_REF} {REPOSITORY_URL} {REPO}
else:
    !git -C {REPO} fetch --depth 1 origin {REPOSITORY_REF}
    !git -C {REPO} checkout --detach FETCH_HEAD
!git -C {REPO} rev-parse HEAD
!cd {REPO} && bash experiments/oa_failure_audit/setup_colab_dft.sh

In [ ]:
BENCHMARK_OUTPUT = SCREEN_ROOT / 'dft_benchmark_two'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/run_dft_hessian.py \
    --subset {SUBSET} --output-dir {BENCHMARK_OUTPUT} --backend gpu4pyscf \
    --max-candidates 2 --resume

In [ ]:
import csv

with SUBSET.open(newline='') as handle:
    subset_rows = list(csv.DictReader(handle))
assert subset_rows, f'DFT subset is empty: {SUBSET}'
with (BENCHMARK_OUTPUT / 'dft_results.csv').open(newline='') as handle:
    benchmark_rows = list(csv.DictReader(handle))
expected_benchmark_count = min(2, len(subset_rows))
failed_benchmark_rows = [row for row in benchmark_rows if row['error']]
required_benchmark_fields = [
    'energy_hartree', 'force_rms_ev_per_angstrom', 'lowest_frequency_cm',
    'second_frequency_cm', 'strict_imaginary_modes', 'wall_seconds',
]
assert len(benchmark_rows) == expected_benchmark_count, (
    f'Expected {expected_benchmark_count} benchmark rows, found {len(benchmark_rows)}'
)
assert not failed_benchmark_rows, f'DFT benchmark failed: {failed_benchmark_rows}'
assert all(
    all(row[field] for field in required_benchmark_fields) for row in benchmark_rows
), benchmark_rows
mean_seconds = sum(float(row['wall_seconds']) for row in benchmark_rows) / len(benchmark_rows)
print(f'Benchmark passed: {len(benchmark_rows)}/{expected_benchmark_count} cases')
print(f'Mean: {mean_seconds / 60:.1f} minutes/case')
print(f'Actual pilot subset: {len(subset_rows)} cases')
print(f'Projected pilot: {mean_seconds * len(subset_rows) / 3600:.1f} serial GPU-hours')
print(f'Projected 96 cases: {mean_seconds * 96 / 3600:.1f} serial GPU-hours')

In [ ]:
RUN_FULL_SUBSET = False  # Change to True only after the two-case benchmark passes.
DFT_OUTPUT = SCREEN_ROOT / 'dft_full'
if RUN_FULL_SUBSET:
    !cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/run_dft_hessian.py \
        --subset {SUBSET} --output-dir {DFT_OUTPUT} --backend gpu4pyscf --resume
    with (DFT_OUTPUT / 'dft_results.csv').open(newline='') as handle:
        full_rows = list(csv.DictReader(handle))
    failed_full_rows = [row for row in full_rows if row['error']]
    assert len(full_rows) == len(subset_rows), (
        f'Expected {len(subset_rows)} full DFT rows, found {len(full_rows)}'
    )
    assert not failed_full_rows, f'Full DFT run has failed rows: {failed_full_rows[:3]}'
    print(f'Full DFT subset passed: {len(full_rows)}/{len(subset_rows)} cases')
else:
    print('Benchmark-only mode. Set RUN_FULL_SUBSET = True after reviewing the timing above.')

## Gate IRC by DFT evidence

Only a stationary DFT index-1 point is immediately IRC-eligible. A nonstationary index-1 raw sample is routed through TS optimization and another Hessian first.

In [ ]:
if RUN_FULL_SUBSET:
    DFT_ENRICHED = SCREEN_ROOT / 'dft_subset_with_results.csv'
    !cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/merge_screening.py \
        --manifest {SUBSET} --screen dft={DFT_OUTPUT / 'dft_results.csv'} \
        --output {DFT_ENRICHED} --overwrite
    IRC_WORKLIST = SCREEN_ROOT / 'irc_worklist.csv'
    !cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/prepare_irc_worklist.py \
        --manifest {DFT_ENRICHED} --output {IRC_WORKLIST} --overwrite

## Package the DFT results for local download

This cell packages all three stages, including completed DFT artifacts and any IRC worklist. After it finishes, use the left **Files** panel to download the archive for local analysis.

In [ ]:
import hashlib
import shutil

bundle_path = Path(shutil.make_archive(
    '/content/oa_audit_03_dft_hessian_generation_8x8_r2_j2',
    'gztar',
    root_dir=OUTPUT_ROOT,
    base_dir='.',
))
digest = hashlib.sha256()
with bundle_path.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
print(f'{bundle_path.name}: {bundle_path.stat().st_size / 1024**2:.2f} MiB')
print(f'sha256: {digest.hexdigest()}')
print(f'Bundle ready at {bundle_path}')
print('Open Files on the left, refresh, then right-click the archive and choose Download.')